# Bab 8: Algoritma Berbasis Pohon dan Metode Ensemble

Bab ini membahas algoritma berbasis pohon, dari satu decision tree yang mudah diinterpretasi hingga metode ensemble yang menggabungkan banyak pohon. Isinya:

1. **Pengantar Decision Tree** -- pemisahan rekursif, Gini impurity, dan interpretabilitas
2. **Random Forest dan Bagging** -- mengurangi variance
3. **Gradient Boosting Machines** -- mengoreksi error secara berurutan untuk mengurangi bias
4. **Tuning Hyperparameter** -- grid search dengan cross-validation
5. **Perbandingan Metode Ensemble** -- bagging vs. boosting vs. stacking

**Dasar matematis:** pada setiap node, decision tree memilih fitur $x_j$ dan threshold $t$ yang memaksimalkan penurunan impurity:

$$\Delta I = I(\text{parent}) - \frac{n_{\text{left}}}{n_{\text{parent}}} I(\text{left}) - \frac{n_{\text{right}}}{n_{\text{parent}}} I(\text{right})$$

dengan $I$ adalah ukuran impurity (Gini atau entropy).

## Persiapan Lingkungan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

print("Semua library berhasil dimuat.")

## 8.1 Pengantar Decision Tree

**Decision tree** membagi ruang fitur lewat serangkaian split biner. Setiap node internal menguji satu fitur, setiap cabang adalah hasil pengujian, dan setiap leaf memberi label kelas (klasifikasi) atau nilai kontinu (regresi). Seluruh proses keputusan bisa ditelusuri dari root ke leaf, sehingga sangat berguna di industri yang diatur regulasi (keuangan, kesehatan).

### Gini Impurity

$$G = 1 - \sum_{c=1}^{C} p_c^2$$

Node murni punya $G = 0$. Untuk 3 kelas (Iris), nilai maksimumnya $1 - 3 \times (1/3)^2 = 0.667$. Alternatifnya adalah **entropy**:

$$H = -\sum_{c=1}^{C} p_c \log_2(p_c)$$

Keduanya biasanya menghasilkan pohon yang mirip; Gini sedikit lebih murah secara komputasi.

In [ ]:
# Muat dataset Iris
iris = load_iris()
X = iris.data
y = iris.target

# Bagi data: 70% train, 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=2024
)

print(f"Training set: {X_train.shape[0]} sampel, {X_train.shape[1]} fitur")
print(f"Test set:     {X_test.shape[0]} sampel, {X_test.shape[1]} fitur")
print(f"Fitur:        {iris.feature_names}")
print(f"Kelas:        {list(iris.target_names)}")
print(f"\nDistribusi kelas (train): {dict(zip(iris.target_names, np.bincount(y_train)))}")
print(f"Distribusi kelas (test):  {dict(zip(iris.target_names, np.bincount(y_test)))}")

**Hasil:** train = 105 sampel, test = 45 sampel, 4 fitur, 3 kelas. Distribusi train cukup seimbang (32 / 38 / 35), sedangkan test agak timpang (setosa 18, versicolor 12, virginica 15). Karena itu kita selalu melihat classification report, bukan akurasi saja. Dengan sampel sekecil ini, split sangat sensitif terhadap data tertentu -- motivasi utama metode ensemble.

In [ ]:
# Buat model dengan hyperparameter default
clf = DecisionTreeClassifier(random_state=2024)

# Latih model
clf.fit(X_train, y_train)

# Prediksi pada test set
y_pred = clf.predict(X_test)

# Evaluasi
accuracy = accuracy_score(y_test, y_pred)
print(f"Akurasi Decision Tree: {accuracy:.4f}")
print(f"Kedalaman Pohon: {clf.get_depth()}")
print(f"Jumlah Leaf: {clf.get_n_leaves()}")

**Hasil:** akurasi **86.67%** (39 dari 45 benar), kedalaman 4, 5 leaf. Pohon seimbang berkedalaman 4 akan punya 16 leaf, jadi 5 leaf menunjukkan split asimetris: setosa terpisah dalam satu split, sedangkan versicolor vs. virginica butuh split lebih dalam. Pohon tanpa batasan rentan **overfitting**.

In [ ]:
# Classification report
report = classification_report(
    y_test, y_pred,
    target_names=iris.target_names,
    output_dict=True
)
report_df = pd.DataFrame(report).transpose()

# Percantik DataFrame
styled_df = (report_df
    .style
    .background_gradient(
        cmap='Blues',
        subset=['precision', 'recall', 'f1-score']
    )
    .format({
        'precision': '{:.3f}',
        'recall': '{:.3f}',
        'f1-score': '{:.3f}',
        'support': '{:.0f}'
    })
)
styled_df

**Ringkasan report (Decision Tree):**

| Kelas | Precision | Recall | F1 |
|:---|:---:|:---:|:---:|
| setosa | 1.000 | 1.000 | 1.000 |
| versicolor | 0.714 | 0.833 | 0.769 |
| virginica | 0.846 | 0.733 | 0.786 |

Setosa terpisah sempurna oleh satu split (petal width $\leq 0.8$ cm). Keenam error ($45 - 39 = 6$) semuanya terjadi di **batas versicolor-virginica**, yang saling tumpang tindih; batas sumbu-sejajar milik pohon tidak mampu memisahkannya dengan sempurna.

### Visualisasi Decision Tree

In [ ]:
plt.figure(figsize=(15, 10))
plot_tree(
    clf,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree untuk Dataset Iris", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('dt_tree.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualisasi decision tree disimpan.")

**Membaca pohon:** root memakai split **petal width $\leq 0.8$ cm** (Gini $\approx 0.664$, mendekati maksimum 0.667) yang langsung memisahkan setosa menjadi leaf murni ($G = 0$). Cabang kanan (73 sampel) dipisah lagi dengan petal length/width untuk versicolor vs. virginica; sebagian leaf tetap tidak murni. Setiap split bisa divalidasi oleh pakar domain, dan inilah nilai interpretabilitas decision tree.

### Mengontrol Kompleksitas Pohon

| Hyperparameter | Fungsi | Efek |
|:---|:---|:---|
| `max_depth` | Kedalaman maksimum | Lebih kecil $\rightarrow$ lebih sederhana |
| `min_samples_split` | Minimum sampel untuk split | Lebih besar $\rightarrow$ split lebih sedikit |
| `min_samples_leaf` | Minimum sampel per leaf | Lebih besar $\rightarrow$ leaf lebih besar |
| `max_features` | Jumlah fitur per split | Lebih kecil $\rightarrow$ lebih acak |
| `min_impurity_decrease` | Penurunan impurity minimum | Lebih besar $\rightarrow$ hanya split kuat |

Semuanya mengatur **trade-off bias-variance**:

$$\text{Expected Error} = \underbrace{\text{Bias}^2}_{\text{underfitting}} + \underbrace{\text{Variance}}_{\text{overfitting}} + \underbrace{\sigma^2}_{\text{noise}}$$

## 8.2 Random Forest dan Bagging

Satu decision tree punya **variance tinggi**. **Random forest** mengatasinya dengan *bootstrap aggregating* (bagging):

1. **Bootstrap sampling:** ambil $B$ sampel acak (dengan pengembalian) dari data latih
2. **Randomisasi fitur:** tiap split hanya mempertimbangkan $\sqrt{p}$ fitur (klasifikasi) atau $p/3$ (regresi)
3. **Agregasi:** majority vote (klasifikasi) atau rata-rata (regresi)

Untuk $B$ pohon dengan variance $\sigma^2$ dan korelasi antar-pohon $\rho$:

$$\text{Var}\left(\frac{1}{B}\sum_{b=1}^{B} T_b(\mathbf{x})\right) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$

Suku kedua mengecil seiring $B$ bertambah; randomisasi fitur menurunkan $\rho$, dan itulah keunggulan random forest dibanding bagging biasa.

In [ ]:
# Random Forest dengan 100 pohon
rf_clf = RandomForestClassifier(n_estimators=100, random_state=2024)

# Latih model
rf_clf.fit(X_train, y_train)

# Prediksi
y_pred_rf = rf_clf.predict(X_test)

# Evaluasi
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Akurasi Random Forest: {accuracy_rf:.4f}")
print(f"Jumlah pohon: {rf_clf.n_estimators}")
print(f"Fitur maksimum per split: {rf_clf.max_features}")

**Hasil:** akurasi **91.11%** (41/45), naik 4.44 poin dari decision tree (86.67%). Tiap pohon melihat $\sqrt{4} = 2$ fitur per split, sehingga setiap pohon sedikit lebih lemah, tetapi error mereka sebagian saling independen dan saling menghilangkan lewat voting (*wisdom of crowds*).

In [ ]:
# Classification report Random Forest
report_rf = classification_report(
    y_test, y_pred_rf,
    target_names=iris.target_names,
    output_dict=True
)
report_rf_df = pd.DataFrame(report_rf).transpose()

styled_rf = (report_rf_df
    .style
    .background_gradient(
        cmap='Blues',
        subset=['precision', 'recall', 'f1-score']
    )
    .format({
        'precision': '{:.3f}',
        'recall': '{:.3f}',
        'f1-score': '{:.3f}',
        'support': '{:.0f}'
    })
)
styled_rf

| Kelas | F1 Decision Tree | F1 Random Forest | Selisih |
|:---|:---:|:---:|:---:|
| setosa | 1.000 | 1.000 | -- |
| versicolor | 0.769 | 0.833 | +0.064 |
| virginica | 0.786 | 0.867 | +0.081 |

Peningkatan seluruhnya datang dari batas versicolor-virginica: voting menghaluskan keputusan batas yang noisy. Recall virginica naik dari 0.733 ke 0.867 (13 vs. 11 dari 15 benar), dan precision/recall menjadi seimbang.

### Analisis Feature Importance

Random forest menghitung pentingnya fitur dari total penurunan Gini impurity yang dikontribusikan tiap fitur, dirata-rata di seluruh pohon.

In [ ]:
# Hitung dan tampilkan feature importance
importances = rf_clf.feature_importances_
indices = np.argsort(importances)[::-1]

print("Peringkat Feature Importance:")
print("-" * 40)
for rank, i in enumerate(indices, 1):
    print(f"  {rank}. {iris.feature_names[i]:20s} {importances[i]:.4f}")

# Plot
plt.figure(figsize=(10, 6))
colors = ['#2196F3' if i >= 2 else '#90CAF9' for i in indices]
plt.bar(
    range(X.shape[1]),
    importances[indices],
    align="center",
    color=colors
)
plt.xticks(
    range(X.shape[1]),
    [iris.feature_names[i] for i in indices],
    rotation=45, ha='right'
)
plt.ylabel("Importance (Penurunan Gini)", fontsize=12)
plt.title("Feature Importance pada Random Forest", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

**Hasil:** petal length (0.4489) dan petal width (0.4360) bersama-sama menyumbang **88.5%** importance; sepal length 0.0901 dan sepal width hanya 0.0251. Ini sesuai pengetahuan botani dan visualisasi pohon sebelumnya.

**Tips produksi:** importance dari random forest sering dipakai untuk *feature selection* -- latih forest, ambil top-$k$ fitur, lalu pakai di model lain. Di sini, model dengan hanya dua fitur petal kemungkinan besar nyaris setara.

## 8.3 Gradient Boosting Machines

**GBM** membangun pohon secara **berurutan**; setiap pohon baru mengoreksi error ensemble sebelumnya. GBM meminimalkan loss $\mathcal{L}(y, F(\mathbf{x}))$ dengan menambah pohon yang mendekati **gradien negatif** loss:

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot h_m(\mathbf{x})$$

dengan $h_m$ pohon yang dilatih pada residual $F_{m-1}$ dan $\eta$ adalah **learning rate**. Berbeda dengan random forest, GBM mengurangi **bias**. Untuk klasifikasi, loss-nya adalah deviance (cross-entropy):

$$\mathcal{L} = -\sum_{i=1}^{N} \sum_{c=1}^{C} y_{i,c} \log(\hat{p}_{i,c})$$

In [ ]:
# GBM dengan 1000 pohon dan learning rate 0.2
gbm_clf = GradientBoostingClassifier(
    n_estimators=1000,
    learning_rate=0.2,
    random_state=2024
)

# Latih model
gbm_clf.fit(X_train, y_train)

# Prediksi
y_pred_gbm = gbm_clf.predict(X_test)

# Evaluasi
accuracy_gbm = accuracy_score(y_test, y_pred_gbm)
print(f"Akurasi GBM: {accuracy_gbm:.4f}")
print(f"Jumlah estimator: {gbm_clf.n_estimators}")
print(f"Learning rate: {gbm_clf.learning_rate}")
print(f"Kedalaman maksimum: {gbm_clf.max_depth}")

**Hasil:** akurasi **86.67%**, sama dengan decision tree dan lebih rendah dari random forest. Dengan 1.000 pohon dan $\eta = 0.2$ (kompleksitas efektif $\approx B \times \eta = 200$) pada hanya 105 sampel, model hampir pasti **overfitting**.

| Properti | Bagging (RF) | Boosting (GBM) |
|:---|:---|:---|
| Konstruksi pohon | Paralel, independen | Berurutan, mengoreksi error |
| Manfaat utama | Kurangi **variance** | Kurangi **bias** |
| Lebih banyak pohon | Selalu membantu/netral | Bisa **overfit** |
| Learning rate | Tidak ada | Hyperparameter kritis |

In [ ]:
# Classification report GBM
report_gbm = classification_report(
    y_test, y_pred_gbm,
    target_names=iris.target_names,
    output_dict=True
)
report_gbm_df = pd.DataFrame(report_gbm).transpose()

styled_gbm = (report_gbm_df
    .style
    .background_gradient(
        cmap='Blues',
        subset=['precision', 'recall', 'f1-score']
    )
    .format({
        'precision': '{:.3f}',
        'recall': '{:.3f}',
        'f1-score': '{:.3f}',
        'support': '{:.0f}'
    })
)
styled_gbm

Metrik per kelas **identik** dengan decision tree (F1 versicolor 0.769, virginica 0.786): GBM yang overfit konvergen ke batas keras yang sama dan melakukan 6 error yang sama. Pelajaran: algoritma yang kuat dengan konfigurasi buruk tidak lebih baik dari baseline sederhana, jadi tuning hyperparameter wajib dilakukan.

## 8.4 Tuning Hyperparameter

`GridSearchCV` mencari kombinasi hyperparameter terbaik secara menyeluruh dengan $k$-fold cross-validation:

$$\text{CV Score} = \frac{1}{k} \sum_{i=1}^{k} \text{Score}(\text{model dilatih pada fold} \neq i, \text{dievaluasi pada fold } i)$$

In [ ]:
# Definisikan grid hyperparameter
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.1, 0.2]
}

# Total kombinasi: 3 x 3 x 3 = 27 konfigurasi x 5 fold = 135 kali fit
print(f"Grid search akan mengevaluasi {3*3*3} kombinasi parameter")
print(f"Dengan CV {5}-fold, totalnya {3*3*3*5} kali fit model")
print(f"\nGrid parameter:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

27 kombinasi $\times$ 5 fold = **135 fit**. Untuk grid besar, `RandomizedSearchCV` atau optimasi Bayesian (mis. Optuna) lebih efisien.

In [ ]:
# Siapkan dan jalankan GridSearchCV
grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=2024),
    param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=True
)

grid_search.fit(X_train, y_train)

# Tampilkan parameter terbaik
print(f"Parameter Terbaik: {grid_search.best_params_}")
print(f"Skor CV Terbaik:   {grid_search.best_score_:.4f}")

**Hasil:** `learning_rate = 0.01`, `max_depth = 4`, `n_estimators = 50`, dengan akurasi CV **98.10%**.

| Parameter | Sebelumnya | Optimal |
|:---|:---:|:---:|
| `learning_rate` | 0.2 | 0.01 (20$\times$ lebih kecil) |
| `n_estimators` | 1.000 | 50 (20$\times$ lebih sedikit) |
| `max_depth` | 3 | 4 |

Kompleksitas efektif turun dari $1000 \times 0.2 = 200$ menjadi $50 \times 0.01 = 0.5$ (400 kali lebih kecil): pada data sekecil ini, ensemble yang konservatif dan belajar pelan bekerja paling baik.

In [ ]:
# Evaluasi model terbaik pada test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)

print(f"Akurasi Test (GBM hasil tuning): {accuracy_best:.4f}")

# Classification report
report_best = classification_report(
    y_test, y_pred_best,
    target_names=iris.target_names,
    output_dict=True
)
report_best_df = pd.DataFrame(report_best).transpose()

styled_best = (report_best_df
    .style
    .background_gradient(
        cmap='Blues',
        subset=['precision', 'recall', 'f1-score']
    )
    .format({
        'precision': '{:.3f}',
        'recall': '{:.3f}',
        'f1-score': '{:.3f}',
        'support': '{:.0f}'
    })
)
styled_best

**Hasil:** akurasi test GBM hasil tuning tetap **86.67%**, padahal skor CV 98.10%. Dengan hanya 45 sampel test, satu kesalahan menggeser akurasi $\approx 2.2$ poin, dan 6 error tadi berada di area tumpang tindih versicolor-virginica. Interval kepercayaan 95%:

$$\hat{p} \pm z_{0.975} \sqrt{\frac{\hat{p}(1-\hat{p})}{n}} = 0.867 \pm 1.96\sqrt{\frac{0.867 \times 0.133}{45}} \approx 0.867 \pm 0.099$$

Akurasi sebenarnya bisa berada di antara **76.8%** dan **96.6%**. Pada dataset kecil, jangan menafsirkan perbedaan akurasi test beberapa poin secara berlebihan; skor CV lebih andal.

### Visualisasi Interaksi Hyperparameter

In [ ]:
# Buat pivot table hasil (dirata-rata terhadap learning_rate)
results = pd.DataFrame(grid_search.cv_results_)
pivot_table = results.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators'
)

print("Rata-rata Akurasi CV berdasarkan max_depth dan n_estimators (dirata-rata terhadap learning_rate):")
print(pivot_table.round(4).to_string())

# Plot heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    pivot_table,
    annot=True,
    fmt='.3f',
    cmap='viridis',
    vmin=0.96,
    vmax=0.985
)
plt.title('Grid Search: Akurasi CV berdasarkan Max Depth dan Jumlah Estimator',
          fontsize=13, fontweight='bold')
plt.xlabel('Jumlah Estimator', fontsize=12)
plt.ylabel('Max Depth', fontsize=12)
plt.tight_layout()
plt.savefig('heatmap_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

**Temuan heatmap:**

- **`max_depth` lebih berpengaruh daripada `n_estimators`.** Dari depth 3 ke 4, akurasi CV naik dari $\approx$ 97.1-97.5% ke **98.1%**; depth 5 tidak menambah apa pun.
- **`n_estimators` menunjukkan diminishing returns.** Pada depth 3, menambah pohon justru menurunkan akurasi sedikit (0.9746 $\rightarrow$ 0.9714); pada depth $\geq$ 4 semuanya 0.9810.
- **Plateau 98.1%** di 6 dari 9 sel menunjukkan model robust terhadap pilihan hyperparameter di rentang itu. Heatmap dengan gradien tajam berarti model rapuh dan perlu tuning hati-hati.

## 8.5 Perbandingan Metode Ensemble

1. **Bagging (Random Forest):** pohon paralel independen, mengurangi variance
2. **Boosting (Gradient Boosting):** pohon berurutan, mengurangi bias
3. **Stacking:** beberapa model dasar yang prediksinya digabung oleh meta-learner

$$\hat{y}_{\text{stack}} = g\left(\hat{y}_{\text{RF}}, \hat{y}_{\text{GBM}}\right)$$

dengan $g$ adalah meta-learner (di sini regresi logistik) yang belajar kapan harus memercayai tiap model dasar.

In [ ]:
# Buat ketiga model ensemble
rf_clf = RandomForestClassifier(n_estimators=100, random_state=2024)
gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=2024)
stacking_clf = StackingClassifier(
    estimators=[('rf', rf_clf), ('gb', gb_clf)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

# Latih semua model
rf_clf.fit(X_train, y_train)
gb_clf.fit(X_train, y_train)
stacking_clf.fit(X_train, y_train)

# Prediksi
rf_pred = rf_clf.predict(X_test)
gb_pred = gb_clf.predict(X_test)
stacking_pred = stacking_clf.predict(X_test)

# Evaluasi
rf_accuracy = accuracy_score(y_test, rf_pred)
gb_accuracy = accuracy_score(y_test, gb_pred)
stacking_accuracy = accuracy_score(y_test, stacking_pred)

print(f"Akurasi Random Forest:     {rf_accuracy:.2f}")
print(f"Akurasi Gradient Boosting: {gb_accuracy:.2f}")
print(f"Akurasi Stacking:          {stacking_accuracy:.2f}")

| Metode | Akurasi | Benar / Total |
|:---|:---:|:---:|
| **Random Forest** (Bagging) | **0.91** | 41 / 45 |
| Gradient Boosting | 0.87 | 39 / 45 |
| Stacking (RF + GBM) | 0.87 | 39 / 45 |

- **Bagging unggul** karena data kecil dan bersih; sumber error utamanya adalah **variance**, yang memang ditangani bagging. Bias sudah rendah, jadi boosting hanya menambah risiko overfitting.
- **Stacking tidak membantu** karena RF dan GBM membuat error yang sama di batas versicolor-virginica, sehingga meta-learner tidak punya informasi pelengkap. Stacking efektif bila model dasarnya beragam (mis. pohon + SVM + neural network).
- **Tips produksi:** pada data tabular kecil, random forest adalah titik awal yang baik (tuning minimal, tahan overfitting, ada feature importance). Gunakan boosting untuk dataset lebih besar, dan stacking hanya untuk model yang benar-benar beragam.

In [ ]:
# Buat bar chart perbandingan
methods = ['Random Forest', 'Gradient Boosting', 'Stacking']
accuracies = [rf_accuracy, gb_accuracy, stacking_accuracy]
colors = ['#2196F3', '#FF7043', '#66BB6A']

plt.figure(figsize=(8, 6))
bars = plt.bar(methods, accuracies, color=colors, edgecolor='white', linewidth=1.5)

# Tambahkan label nilai di atas bar
for bar, acc in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.005,
        f'{acc:.2f}', ha='center', va='bottom', fontsize=14, fontweight='bold'
    )

plt.title('Perbandingan Akurasi Metode Ensemble',
          fontsize=14, fontweight='bold')
plt.ylabel('Akurasi', fontsize=12)
plt.ylim(0.7, 1.0)
plt.axhline(y=accuracy, color='gray', linestyle='--', alpha=0.5,
            label=f'Single Decision Tree ({accuracy:.2f})')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

Garis putus-putus abu-abu adalah baseline decision tree (0.87). Hanya random forest yang melampauinya; **ensemble tidak dijamin lebih baik dari model tunggal**, terutama pada dataset kecil.

## Ringkasan Bab

| Algoritma | Akurasi Test | Strategi | Cocok Untuk |
|:---|:---:|:---|:---|
| Decision Tree | 86.67% | Partisi rekursif tunggal | Interpretabilitas, baseline cepat |
| Random Forest | **91.11%** | Bagging paralel + randomisasi fitur | Masalah dominan variance, default yang kokoh |
| GBM (belum tuning) | 86.67% | Boosting berurutan | Dataset besar dengan pola kompleks |
| GBM (hasil tuning) | 86.67% | Boosting hasil tuning | Bila tuning dimungkinkan |
| Stacking | 86.67% | Meta-learning atas model beragam | Kombinasi model yang beragam |

**Poin penting:**

1. **Trade-off bias-variance adalah inti metode berbasis pohon.** Decision tree: bias rendah, variance tinggi; random forest mengurangi variance; GBM mengurangi bias.
2. **Tuning wajib untuk boosting.** GBM default ($B = 1000$, $\eta = 0.2$) hanya setara decision tree, sedangkan konfigurasi optimal ($B = 50$, $\eta = 0.01$) mencapai 98.1% di CV.
3. **Lebih kompleks belum tentu lebih baik.** Pada Iris (150 sampel), random forest mengalahkan boosting dan stacking.
4. **Feature importance menuntun feature engineering.** Fitur petal menyumbang 88.5% kekuatan keputusan random forest.